# TePMA — Fine-tuning Qwen3-1.7B 🎙️
### Teaching a small model to run resume interviews in English, Hindi & Punjabi

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** (free tier is fine). Not TPU — this notebook
   uses Unsloth, which is CUDA-only.
2. Run `TePMA_data_colab.ipynb` first. It generates the training data and writes it to Drive,
   and it records the **baseline scores** this run has to beat.

---

## What we are doing, in one picture

```
  Qwen3-8B on a Colab GPU  ──teaches──>  Qwen3-1.7B here
     (the teacher)                          (the student)
      slow, 5 GB                       fast, 1.1 GB, just as good AT THIS TASK
```

This is called **distillation**. The teacher generated the fake interviews with gold JSON
answers in the data notebook. The student learns to imitate them. Because the student only
has to be good at *your* task — not at poetry, physics and coding — a much smaller model
suffices.

**Why bother:** in the running app, every interviewer reply is time the user spends sitting
in silence waiting for the kiosk to speak. That latency is the whole point of this exercise.

## What we are deliberately NOT doing

We are **not** teaching the model Indian universities, PIN codes, or today's date.
Those live in `facts.py` on your Mac as a lookup table.

> **The rule to remember: fine-tuning teaches SKILLS, not FACTS.**
> A model trained on a list of universities does not *know* them — it learns to produce
> plausible-sounding university names, which is hallucination. A 40-line fuzzy matcher
> fixes `"Thapadi University"` → `"Thapar Institute..."` correctly 100% of the time.

Read `finetune/GUIDE.md` for the full reasoning.

---
# Step 1 — Install Unsloth

**Unsloth** rewrites the training maths to be ~2× faster and use ~60% less GPU memory than
the standard HuggingFace stack. That is the difference between "fits on a free T4" and
"needs an A100 you have to pay for".

In [ ]:
%%capture
!pip install unsloth

---
# Step 2 — Load your dataset from Drive

The data notebook (`TePMA_data_colab.ipynb`) already wrote `train.jsonl` and `eval.jsonl` to
Drive, so there is nothing to upload by hand — and nothing to re-upload when this runtime
disconnects mid-training.

Each line is one training example in **chat format**:

```json
{"messages": [
   {"role": "system",    "content": "You are a resume interviewer..."},
   {"role": "user",      "content": "..."},
   {"role": "assistant", "content": "the reply the model should learn"}
]}
```

The dataset mixes both jobs deliberately — interviewer turns *and* transcript→JSON
extraction — so one model learns both.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORK = Path('/content/drive/MyDrive/tepma')
DATA = WORK / 'data'
TRAIN, EVAL = DATA / 'train.jsonl', DATA / 'eval.jsonl'

assert TRAIN.exists() and EVAL.exists(), (
    f'{DATA} is missing train.jsonl/eval.jsonl — run TePMA_data_colab.ipynb first')

print('train:', sum(1 for _ in TRAIN.open()), 'examples')
print('eval: ', sum(1 for _ in EVAL.open()),  'examples')

---
# Step 3 — Load the base model in 4-bit

Two ideas at work here:

**Quantisation** — we store the model's weights using 4 bits per number instead of 16.
The model shrinks ~4× in memory with almost no quality loss. These weights stay **frozen**;
we never change them.

**Why Qwen3-1.7B?** It is small enough to be fast, and its pre-training included substantial
Hindi and other Indic-language data — which matters for your three-language requirement.
A model with no Hindi exposure could not learn Hindi from 400 examples.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-1.7B",
    max_seq_length = 4096,    # must fit the longest transcript + JSON answer
    load_in_4bit   = True,    # <- the "Q" in QLoRA
)
print(torch.cuda.get_device_name(0))

---
# Step 4 — Attach LoRA adapters

Instead of training all 1.7 billion parameters, we freeze them and bolt small trainable
matrices onto the important layers:

```
      frozen weight W  (1.7B parameters, untouched)
             │
   input ────┼──────────────────> output
             │                       ▲
             └──> A ──> B ───────────┘
                  trainable (~20M = ~1%)
```

`A` starts at zero, so training begins as an exact copy of the original model and learns a
small **correction** on top. This is why LoRA is cheap and why it rarely destroys the
model's existing abilities (like its Hindi).

| Knob | What it does | When to raise it |
|---|---|---|
| `r=16` | adapter capacity | 32 if the model underfits |
| `lora_alpha=32` | adapter influence | keep at `2*r` |
| `lora_dropout=0` | regularisation | >0 only if overfitting badly |

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",   # saves a lot of memory
    random_state = 7,
)
model.print_trainable_parameters()   # <- look how small the trainable % is

---
# Step 5 — Apply the chat template

The model never sees `{"role": "user"}`. It sees one long string with special marker tokens:

```
<|im_start|>system
You are a resume interviewer...<|im_end|>
<|im_start|>user
mera naam Aditya hai<|im_end|>
<|im_start|>assistant
Aap kaunse college se hain?<|im_end|>
```

`apply_chat_template` does that conversion. **`enable_thinking=False` is critical** —
Qwen3 can emit `<think>...</think>` reasoning blocks, and TePMA calls Ollama with
`think: false`. Training format must match production format exactly, or the model will
produce `<think>` tags your app then has to strip.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": str(TRAIN), "eval": str(EVAL)})

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False,
        add_generation_prompt=False, enable_thinking=False)}

ds = ds.map(to_text, remove_columns=["messages"])
print(ds["train"][0]["text"][:800])   # this exact string is what the model trains on

---
# Step 6 — Train

### The one non-obvious line: `train_on_responses_only`

Without it, the model is graded on predicting *every* token — including your long system
prompt and the candidate's answers. That wastes capacity on things it never needs to
generate. This wrapper **masks the loss** so the model is scored **only** on the assistant
replies (the interview question, or the JSON). Big accuracy win for free.

### Choosing epochs
- **1 epoch** — usually underfits on a few hundred samples
- **2 epochs** — the sweet spot here ✅
- **5+ epochs** — memorises your data, gets worse on anything new (**overfitting**)

### What to watch in the output
`train_loss` should fall steadily. `eval_loss` (printed once per epoch) should fall too.

```
 loss                                  loss
   │╲___ train      HEALTHY              │╲        ___/ eval   OVERFITTING
   │ ╲___ eval      (both fall)          │ ╲___ ___/           (eval turns up:
   └────────> steps                      │  ╲__/  train         stop earlier)
                                         └────────> steps
```

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset  = ds["eval"],
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,     # effective batch size = 16
        num_train_epochs = 2,
        learning_rate = 2e-4,                # LoRA likes a high LR
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        logging_steps = 5,
        eval_strategy = "epoch",
        save_strategy = "epoch",             # checkpoint to Drive: a Colab
        output_dir = str(WORK / "outputs"),  # disconnect costs one epoch, not the run
        save_total_limit = 2,
        optim = "adamw_8bit",
        report_to = "none",
        seed = 7,
    ),
)

# grade the model ONLY on what it should generate
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# If this run died partway, re-run this cell with resume_from_checkpoint=True instead.
stats = trainer.train()

---
# Step 7 — Sanity check in all three languages

Before exporting a 1 GB file, spend 30 seconds checking the model actually behaves.
We test the thing that is hardest and most likely to break: **replying in the
candidate's language**.

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = ("You are a friendly, professional resume interviewer on a voice call. "
          "Keep replies short and natural to say aloud. Ask ONE question at a time. "
          "Plain text only. Always reply in the language the candidate is mainly using.")

def ask(user_line, opener="Hello! Could you tell me your name and the role you are targeting?"):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":"(The candidate has joined the voice call. Greet them and begin the interview.)"},
            {"role":"assistant","content":opener},
            {"role":"user","content":user_line}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        enable_thinking=False, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=80, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

for line in [
    "hi umm im rahul and im looking for data analyst jobs",
    "मेरा नाम प्रिया है और मैं सॉफ्टवेयर इंजीनियर की जॉब ढूंढ रही हूं",
    "ਸਤ ਸ੍ਰੀ ਅਕਾਲ, ਮੇਰਾ ਨਾਮ ਗੁਰਪ੍ਰੀਤ ਹੈ",
    "mera naam Aditya hai, I'm doing B.Tech from Chitkara",
]:
    print("CANDIDATE:", line)
    print("MODEL    :", ask(line))
    print("-" * 70)

**What good looks like:** each reply is in the *same language* as the candidate, is one
short question, and contains no markdown, no emoji, no `<think>` block.

If Punjabi comes back in Hindi or English → you need more Punjabi samples in the dataset.
Go back to `generate_data.py`, raise the `pa` weight in `LANGUAGES`, generate more, retrain.

---
# Step 8 — Export to GGUF for Ollama

**GGUF** is the file format llama.cpp and Ollama use. This cell merges the LoRA adapters
into the base weights (producing one standalone model) and quantises it to 4-bit.

`q4_k_m` is the recommended quantisation: ~1.1 GB, the best quality-per-byte trade-off.
This cell takes ~10 minutes.

In [ ]:
model.save_pretrained_gguf("tepma-model", tokenizer, quantization_method="q4_k_m")
!ls -lh tepma-model/*.gguf

In [ ]:
# Copy to Drive rather than files.download() — a ~1 GB browser download from Colab
# usually stalls, and the data notebook expects to find it here for scoring.
import glob, shutil

gguf = glob.glob("tepma-model/*.gguf")[0]
dest = WORK / "tepma.gguf"
shutil.copy(gguf, dest)
print("saved:", dest, f"({dest.stat().st_size / 1e9:.2f} GB)")

---
# Step 9 — Score it, then measure latency on the Mac

Two different questions, answered in two different places.

### Accuracy — back in the data notebook

`tepma.gguf` is now on Drive. Open `TePMA_data_colab.ipynb` → **Step 6** and run it: it
registers the GGUF with Ollama and scores it on the same held-out interviews as the
baselines, so the numbers line up directly.

| Outcome | Meaning | Action |
|---|---|---|
| Within ~5 pts of the 8B | 🎉 it worked | go measure latency below |
| Much lower | underfitting | more data → 3 epochs → `r=32` |
| Great train loss, poor eval | overfitting | fewer epochs, more data |
| `no_hallucination` dropped | inventing jobs | do **not** ship — more data, and keep the 8B for extraction |

### Latency — only the Mac can tell you this

Colab timings are meaningless for a kiosk. Download `tepma.gguf` from Drive, then locally:

```bash
cd ~/Codes/curin/projects/TePMA/finetune

cat > Modelfile <<'EOF'
FROM ./tepma.gguf
PARAMETER temperature 0.7
PARAMETER num_ctx 4096
EOF

ollama create tepma -f Modelfile

.venv/bin/python finetune/eval_model.py --model tepma      # your fine-tuned 1.7B
.venv/bin/python finetune/eval_model.py --model qwen3:8b   # the teacher, for reference
```

Compare the `interview reply` medians. If `tepma` is several times faster at comparable
accuracy, set `LLM_MODEL=tepma` in `.env` — no code change needed.

**Remember what you are measuring.** Loss is a training signal. *Field accuracy on unseen
interviews* decides whether users get a correct resume; *reply latency on the Mac* decides
whether they enjoy using it.